## jul16 — Daily TE, Sep 16 → Oct 21

### Goal
Prof. Dodson actually wants daily TE < 0.1%. Expiry-to-expiry sidestepped this.
Testing whether the official roll formula + correct timestamps can actually
bring daily TE down.

### Checking
1. Roll handled as (1+Ra)(1+Rb)(1+Rc), not naive differencing
2. C_t = last bid/ask before 4pm (not our old 15:45 snapshot)
3. Does the held contract go stale as it drifts deep ITM/OTM by month-end
4. (later) equity basket issue — parked for now

### Test window
Sep 16 roll (K=3855, VWAP=$118.53) → Oct 21 roll

In [5]:
# ── Build Daily Option + SPX Series: Sep 16 -> Oct 21, 2022 ──────────────────
import pandas as pd
import numpy as np
import os
import yfinance as yf

DATA_DIR = "../data/raw/cboe_spx_2022/"
HELD_STRIKE = 3855
HELD_EXPIRATION = "2022-10-21"   # the contract sold at the Sep 16 roll

# Trading days from Sep 16 (roll) through Oct 21 (next roll), inclusive
date_range = pd.bdate_range("2022-09-16", "2022-10-21")

rows = []
for d in date_range:
    fname = f"UnderlyingOptionsTradesCalcs_{d.strftime('%Y-%m-%d')}.csv"
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Missing file for {d.date()}, skipping")
        continue

    df = pd.read_csv(fpath)
    df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

    # Filter to the ONE held contract: SPX call, strike=3855, exp=Oct 21
    mask = (
        (df["root"] == "SPX")
        & (df["option_type"] == "C")
        & (df["strike"] == HELD_STRIKE)
        & (df["expiration"] == HELD_EXPIRATION)
    )
    day_df = df[mask].sort_values("quote_datetime")

    if day_df.empty:
        print(f"No quotes for held contract on {d.date()}")
        continue

    # "Last bid/ask reported before 4:00 PM ET" per official methodology
    before_close = day_df[day_df["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
    if before_close.empty:
        before_close = day_df  # fallback

    last_quote = before_close.iloc[-1]
    C_t = (last_quote["best_bid"] + last_quote["best_ask"]) / 2

    rows.append({
        "date": d.date(),
        "C_t": C_t,
        "last_quote_time": last_quote["quote_datetime"],
        "n_quotes_before_4pm": len(before_close),
    })

option_series = pd.DataFrame(rows)
print(option_series)

# ── S&P 500 official close (NOT underlying_bid from options tick data) ──────
spx = yf.download("^GSPC", start="2022-09-15", end="2022-10-22")
spx_close = spx["Close"].reset_index()
spx_close.columns = ["date", "S_t"]
spx_close["date"] = spx_close["date"].dt.date

print(spx_close)

No quotes for held contract on 2022-10-21
          date      C_t         last_quote_time  n_quotes_before_4pm
0   2022-09-16  129.600 2022-09-16 15:59:43.060                 1103
1   2022-09-19  136.800 2022-09-19 15:28:04.723                   17
2   2022-09-20  115.850 2022-09-20 15:59:00.365                   15
3   2022-09-21   88.450 2022-09-21 15:46:07.726                    5
4   2022-09-22   63.650 2022-09-22 15:59:39.543                   16
5   2022-09-23   41.800 2022-09-23 15:59:00.408                  178
6   2022-09-26   36.600 2022-09-26 15:13:03.364                   11
7   2022-09-27   28.950 2022-09-27 15:59:50.981                   26
8   2022-09-28   47.300 2022-09-28 15:56:07.734                   22
9   2022-09-29   23.500 2022-09-29 12:32:24.966                    2
10  2022-09-30   12.250 2022-09-30 15:57:33.077                   11
11  2022-10-03   21.100 2022-10-03 15:56:33.691                   15
12  2022-10-04   52.750 2022-10-04 15:58:50.030              

[*********************100%***********************]  1 of 1 completed

          date          S_t
0   2022-09-15  3901.350098
1   2022-09-16  3873.330078
2   2022-09-19  3899.889893
3   2022-09-20  3855.929932
4   2022-09-21  3789.929932
5   2022-09-22  3757.989990
6   2022-09-23  3693.229980
7   2022-09-26  3655.040039
8   2022-09-27  3647.290039
9   2022-09-28  3719.040039
10  2022-09-29  3640.469971
11  2022-09-30  3585.620117
12  2022-10-03  3678.429932
13  2022-10-04  3790.929932
14  2022-10-05  3783.280029
15  2022-10-06  3744.520020
16  2022-10-07  3639.659912
17  2022-10-10  3612.389893
18  2022-10-11  3588.840088
19  2022-10-12  3577.030029
20  2022-10-13  3669.909912
21  2022-10-14  3583.070068
22  2022-10-17  3677.949951
23  2022-10-18  3719.979980
24  2022-10-19  3695.159912
25  2022-10-20  3665.780029
26  2022-10-21  3752.750000


In [6]:
# ── Merge option + SPX series ──
merged = option_series.merge(spx_close, on="date", how="inner")
merged["moneyness"] = spx_close.set_index("date").loc[merged["date"], "S_t"].values - HELD_STRIKE
merged["portfolio_value"] = merged["S_t"] - merged["C_t"]

print(merged[["date", "S_t", "C_t", "moneyness", "portfolio_value", "n_quotes_before_4pm"]])

# ── Non-roll-date daily return: (S_t - C_t) / (S_t-1 - C_t-1) ──
merged["ret"] = merged["portfolio_value"] / merged["portfolio_value"].shift(1) - 1
print("\nDaily returns (non-roll formula):")
print(merged[["date", "ret"]])
print(f"\nAnnualized std of this segment: {merged['ret'].std() * (252**0.5):.4%}")

          date          S_t      C_t   moneyness  portfolio_value  \
0   2022-09-16  3873.330078  129.600   18.330078      3743.730078   
1   2022-09-19  3899.889893  136.800   44.889893      3763.089893   
2   2022-09-20  3855.929932  115.850    0.929932      3740.079932   
3   2022-09-21  3789.929932   88.450  -65.070068      3701.479932   
4   2022-09-22  3757.989990   63.650  -97.010010      3694.339990   
5   2022-09-23  3693.229980   41.800 -161.770020      3651.429980   
6   2022-09-26  3655.040039   36.600 -199.959961      3618.440039   
7   2022-09-27  3647.290039   28.950 -207.709961      3618.340039   
8   2022-09-28  3719.040039   47.300 -135.959961      3671.740039   
9   2022-09-29  3640.469971   23.500 -214.530029      3616.969971   
10  2022-09-30  3585.620117   12.250 -269.379883      3573.370117   
11  2022-10-03  3678.429932   21.100 -176.570068      3657.329932   
12  2022-10-04  3790.929932   52.750  -64.070068      3738.179932   
13  2022-10-05  3783.280029   46.5

In [7]:
# ── Load official BXM daily levels, compute actual TE ──
bxm = pd.read_csv("../data/raw/BXM_History.csv")
bxm["date"] = pd.to_datetime(bxm["DATE"], format="%m/%d/%Y").dt.date
bxm = bxm[(bxm["date"] >= merged["date"].min()) & (bxm["date"] <= merged["date"].max())].sort_values("date")

bxm["bxm_ret"] = bxm["BXM"] / bxm["BXM"].shift(1) - 1

# ── Merge our reconstruction with official BXM ──
te_check = merged[["date", "ret"]].merge(bxm[["date", "bxm_ret"]], on="date", how="inner")
te_check["te"] = te_check["ret"] - te_check["bxm_ret"]

print(te_check)
print(f"\nOur reconstruction ann. vol:  {te_check['ret'].std() * (252**0.5):.4%}")
print(f"Official BXM ann. vol:        {te_check['bxm_ret'].std() * (252**0.5):.4%}")
print(f"Daily TE (ours - official):   {te_check['te'].std() * (252**0.5):.4%}")
print(f"Mean daily TE:                {te_check['te'].mean():.4%}")

          date       ret   bxm_ret        te
0   2022-09-16       NaN       NaN       NaN
1   2022-09-19  0.005171  0.003286  0.001885
2   2022-09-20 -0.006115 -0.005317 -0.000797
3   2022-09-21 -0.010321 -0.008528 -0.001792
4   2022-09-22 -0.001929 -0.003171  0.001242
5   2022-09-23 -0.011615 -0.011876  0.000261
6   2022-09-26 -0.009035 -0.008147 -0.000887
7   2022-09-27 -0.000028 -0.000778  0.000750
8   2022-09-28  0.014758  0.015952 -0.001194
9   2022-09-29 -0.014917 -0.015543  0.000626
10  2022-09-30 -0.012054 -0.012259  0.000205
11  2022-10-03  0.023496  0.023327  0.000169
12  2022-10-04  0.022106  0.021979  0.000127
13  2022-10-05 -0.000374 -0.000520  0.000145
14  2022-10-06 -0.007482 -0.006557 -0.000925
15  2022-10-07 -0.021519 -0.021418 -0.000101
16  2022-10-10 -0.006495 -0.006458 -0.000037
17  2022-10-11 -0.006019 -0.006197  0.000178
18  2022-10-12 -0.003212 -0.002864 -0.000348
19  2022-10-13  0.025497  0.024660  0.000836
20  2022-10-14 -0.022259 -0.021549 -0.000710
21  2022-1

In [8]:
# ── Add SPX dividend points via Price Index vs Total Return Index gap ──
spx_tr = yf.download("^SP500TR", start="2022-09-15", end="2022-10-22")["Close"].reset_index()
spx_tr.columns = ["date", "SPX_TR"]
spx_tr["date"] = spx_tr["date"].dt.date

div_check = spx_close.merge(spx_tr, on="date")
div_check["tr_ret"] = div_check["SPX_TR"] / div_check["SPX_TR"].shift(1) - 1
div_check["price_ret"] = div_check["S_t"] / div_check["S_t"].shift(1) - 1
div_check["implied_div_yield"] = div_check["tr_ret"] - div_check["price_ret"]

print(div_check[["date", "implied_div_yield"]])

[*********************100%***********************]  1 of 1 completed

          date  implied_div_yield
0   2022-09-15                NaN
1   2022-09-16       1.647088e-05
2   2022-09-19       1.078311e-05
3   2022-09-20       9.900394e-06
4   2022-09-21       6.568807e-05
5   2022-09-22       4.619828e-05
6   2022-09-23       2.316222e-05
7   2022-09-26       1.877055e-06
8   2022-09-27       6.304372e-05
9   2022-09-28       5.881752e-05
10  2022-09-29       1.410976e-04
11  2022-09-30       4.553274e-05
12  2022-10-03       8.020722e-06
13  2022-10-04       8.937624e-05
14  2022-10-05       9.737888e-05
15  2022-10-06       2.523000e-04
16  2022-10-07       3.278522e-05
17  2022-10-10      -9.396329e-07
18  2022-10-11       1.698260e-05
19  2022-10-12       2.692844e-05
20  2022-10-13       1.330097e-04
21  2022-10-14       3.730436e-05
22  2022-10-17       1.613140e-06
23  2022-10-18       2.121540e-05
24  2022-10-19      -2.855356e-06
25  2022-10-20       1.125567e-04
26  2022-10-21       1.957347e-05


In [9]:
# ── Pull missing pieces for the Oct 21 roll leg ──
roll_file = os.path.join(DATA_DIR, "UnderlyingOptionsTradesCalcs_2022-10-21.csv")
roll_df = pd.read_csv(roll_file)
roll_df["quote_datetime"] = pd.to_datetime(roll_df["quote_datetime"])

# S_VWAV: underlying VWAP during the new option's VWAP window (11:30-1:30 ET)
window = roll_df[
    (roll_df["quote_datetime"].dt.time >= pd.Timestamp("11:30:00").time())
    & (roll_df["quote_datetime"].dt.time <= pd.Timestamp("13:30:00").time())
]
S_VWAV = window["underlying_bid"].mean()  # simple approx, refine to trade-weighted later if needed
print(f"S_VWAV (Oct 21, approx): {S_VWAV:.4f}")

# C_t (new): mid-quote of the NEW contract (K=3680, exp Nov 18) before 4pm on Oct 21
new_mask = (
    (roll_df["root"] == "SPX")
    & (roll_df["option_type"] == "C")
    & (roll_df["strike"] == 3680)
    & (roll_df["expiration"] == "2022-11-18")
)
new_day = roll_df[new_mask].sort_values("quote_datetime")
before_4pm = new_day[new_day["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
C_t_new = (before_4pm.iloc[-1]["best_bid"] + before_4pm.iloc[-1]["best_ask"]) / 2
print(f"C_t (new contract, Oct 21 close): {C_t_new:.4f}")

# ── Compute Ra, Rb, Rc ──
S_prev, C_prev = 3665.780029, 0.025   # Oct 20
S_SOQ, K_old = 3656.28, 3855
C_settle = max(0, S_SOQ - K_old)
C_VWAP = 135.64
S_t = 3752.750000

Ra = (S_SOQ + 0 - C_settle) / (S_prev - C_prev) - 1
Rb = S_VWAV / S_SOQ - 1
Rc = (S_t - C_t_new) / (S_VWAV - C_VWAP) - 1

roll_return = (1 + Ra) * (1 + Rb) * (1 + Rc) - 1
naive_return = (S_t - C_t_new) / (S_prev - C_prev) - 1  # what we'd get WITHOUT the 3-leg formula

print(f"\nRa={Ra:.6f}  Rb={Rb:.6f}  Rc={Rc:.6f}")
print(f"Proper 3-leg roll return: {roll_return:.4%}")
print(f"Naive (non-roll formula applied at roll boundary): {naive_return:.4%}")

# ── Compare against official BXM ──
bxm_oct21 = bxm[bxm["date"] == pd.Timestamp("2022-10-21").date()]
print(f"Official BXM return on Oct 21: {bxm_oct21['bxm_ret'].values}")

S_VWAV (Oct 21, approx): 3713.0182
C_t (new contract, Oct 21 close): 160.9500

Ra=-0.002585  Rb=0.015518  Rc=0.004031
Proper 3-leg roll return: 1.6977%
Naive (non-roll formula applied at roll boundary): -2.0175%
Official BXM return on Oct 21: []


In [10]:
# ── Fix date comparison bug ──
bxm_oct21 = bxm[bxm["date"] == pd.Timestamp("2022-10-21").date()]
print(bxm_oct21)  # debug: check dtype mismatch

# bxm["date"] is a date object (from .dt.date), compare with a date object not Timestamp
target_date = pd.Timestamp("2022-10-21").date()
bxm_oct21 = bxm[bxm["date"] == target_date]
print(f"\nOfficial BXM return on Oct 21: {bxm_oct21['bxm_ret'].values}")
print(f"\nProper 3-leg roll return:  1.6977%")
print(f"Naive roll return:        -2.0175%")

Empty DataFrame
Columns: [DATE, BXM, date, bxm_ret]
Index: []

Official BXM return on Oct 21: []

Proper 3-leg roll return:  1.6977%
Naive roll return:        -2.0175%


In [11]:
# ── Reload BXM without the truncated range from `merged` ──
bxm_full = pd.read_csv("../data/raw/BXM_History.csv")
bxm_full["date"] = pd.to_datetime(bxm_full["DATE"], format="%m/%d/%Y").dt.date
bxm_full = bxm_full.sort_values("date").reset_index(drop=True)
bxm_full["bxm_ret"] = bxm_full["BXM"] / bxm_full["BXM"].shift(1) - 1

target_date = pd.Timestamp("2022-10-21").date()
row = bxm_full[bxm_full["date"] == target_date]
print(row)

            DATE      BXM        date   bxm_ret
5180  10/21/2022  1536.47  2022-10-21  0.016581


## Extend to full year (11 roll-to-roll periods, Jan21 -> Dec16)

### Plan
1. Build master roll table: SOQ, K_new, K_old, C_VWAP_new for all 12 roll dates
2. Reusable function: daily option mid-quote series for a held contract
3. Reusable function: roll-day leg (Ra, Rb, Rc) using official formula
4. Loop over 11 periods, concatenate into one daily return series
5. Compare full-year daily TE against BXM_History.csv

In [12]:
# ── Master roll table (12 roll dates, 2022) ──
roll_dates = pd.DataFrame({
    "roll_date": ["2022-01-21","2022-02-18","2022-03-18","2022-04-14",
                  "2022-05-20","2022-06-17","2022-07-15","2022-08-19",
                  "2022-09-16","2022-10-21","2022-11-18","2022-12-16"],
    "SOQ": [4472.07, 4383.70, 4409.35, 4452.07, 3937.64, 3663.76,
            3839.81, 4258.21, 3871.24, 3656.28, 3983.42, 3871.47],
    "K_new": [4475, 4365, 4415, 4430, 3885, 3655, 3850, 4235, 3855, 3680, 3950, 3850],
    "C_VWAP_new": [89.06, 104.36, 99.99, 92.51, 97.64, 122.02,
                   107.05, 85.29, 118.53, 135.64, 96.31, 97.48],
})
roll_dates["roll_date"] = pd.to_datetime(roll_dates["roll_date"]).dt.date
roll_dates["K_old"] = roll_dates["K_new"].shift(1)
roll_dates["C_settle"] = np.maximum(0, roll_dates["SOQ"] - roll_dates["K_old"])
# expiration of the K_new contract = next roll date
roll_dates["expiration"] = roll_dates["roll_date"].shift(-1)

# ── Patch: Dec 16 roll's new contract expires Jan 20, 2023 (3rd Friday, not in our table) ──
roll_dates.loc[roll_dates["roll_date"] == pd.Timestamp("2022-12-16").date(), "expiration"] = pd.Timestamp("2023-01-20").date()

print(roll_dates)

     roll_date      SOQ  K_new  C_VWAP_new   K_old  C_settle  expiration
0   2022-01-21  4472.07   4475       89.06     NaN       NaN  2022-02-18
1   2022-02-18  4383.70   4365      104.36  4475.0      0.00  2022-03-18
2   2022-03-18  4409.35   4415       99.99  4365.0     44.35  2022-04-14
3   2022-04-14  4452.07   4430       92.51  4415.0     37.07  2022-05-20
4   2022-05-20  3937.64   3885       97.64  4430.0      0.00  2022-06-17
5   2022-06-17  3663.76   3655      122.02  3885.0      0.00  2022-07-15
6   2022-07-15  3839.81   3850      107.05  3655.0    184.81  2022-08-19
7   2022-08-19  4258.21   4235       85.29  3850.0    408.21  2022-09-16
8   2022-09-16  3871.24   3855      118.53  4235.0      0.00  2022-10-21
9   2022-10-21  3656.28   3680      135.64  3855.0      0.00  2022-11-18
10  2022-11-18  3983.42   3950       96.31  3680.0    303.42  2022-12-16
11  2022-12-16  3871.47   3850       97.48  3950.0      0.00  2023-01-20


In [36]:
RARE_CODES = [136, 13, 118, 137, 40, 41, 42]

def get_option_daily_series(strike, expiration, start_date, end_date):
    valid_days = [d for d in spx_year.index if start_date.date() <= d <= end_date.date()]
    rows = []
    last_known_C, last_known_delta, last_known_S = None, None, None

    for d in valid_days:
        fname = f"UnderlyingOptionsTradesCalcs_{d.strftime('%Y-%m-%d')}.csv"
        fpath = os.path.join(DATA_DIR, fname)
        S_today = spx_year.loc[d]

        def carry_forward():
            if last_known_C is None:
                return None
            return max(0, last_known_C + last_known_delta * (S_today - last_known_S))

        if not os.path.exists(fpath):
            C_approx = carry_forward()
            if C_approx is not None:
                rows.append({"date": d, "C_t": C_approx, "n_quotes": 0, "approx": True})
            continue

        df = pd.read_csv(fpath)
        df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])
        mask = (
            (df["root"] == "SPX") & (df["option_type"] == "C")
            & (df["strike"] == strike)
            & (df["expiration"] == expiration.strftime("%Y-%m-%d"))
            & (~df["trade_condition_id"].isin(RARE_CODES))
        )
        day_df = df[mask].sort_values("quote_datetime")

        if day_df.empty:
            C_approx = carry_forward()
            if C_approx is not None:
                rows.append({"date": d, "C_t": C_approx, "n_quotes": 0, "approx": True})
            continue

        before_4pm = day_df[day_df["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
        if before_4pm.empty:
            before_4pm = day_df
        last_q = before_4pm.iloc[-1]
        last_known_C = (last_q["best_bid"] + last_q["best_ask"]) / 2
        last_known_delta = last_q["trade_delta"]
        last_known_S = S_today
        rows.append({"date": d, "C_t": last_known_C, "n_quotes": len(before_4pm), "approx": False})

    return pd.DataFrame(rows)

    
# ── compute_roll_leg: official Ra x Rb x Rc formula, expiration-filtered, trade-weighted S_VWAV ──
def compute_roll_leg(roll_date, K_old, K_new, K_new_expiration, SOQ, C_settle, C_VWAP_new, S_prev, C_prev):
    fname = f"UnderlyingOptionsTradesCalcs_{roll_date.strftime('%Y-%m-%d')}.csv"
    fpath = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(fpath)
    df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

    new_mask = (
        (df["root"] == "SPX") & (df["option_type"] == "C")
        & (df["strike"] == K_new)
        & (df["expiration"] == K_new_expiration.strftime("%Y-%m-%d"))
    )
    new_day = df[new_mask].sort_values("quote_datetime")

    window_mask = (
        (new_day["quote_datetime"].dt.time >= pd.Timestamp("11:30:00").time())
        & (new_day["quote_datetime"].dt.time <= pd.Timestamp("13:30:00").time())
    )
    window_trades = new_day[window_mask]
    S_VWAV = (window_trades["underlying_bid"] * window_trades["trade_size"]).sum() / window_trades["trade_size"].sum()

    before_4pm = new_day[new_day["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
    C_t_new = (before_4pm.iloc[-1]["best_bid"] + before_4pm.iloc[-1]["best_ask"]) / 2

    S_t = spx_year.loc[roll_date]

    Ra = (SOQ + 0 - C_settle) / (S_prev - C_prev) - 1
    Rb = S_VWAV / SOQ - 1
    Rc = (S_t - C_t_new) / (S_VWAV - C_VWAP_new) - 1
    roll_ret = (1 + Ra) * (1 + Rb) * (1 + Rc) - 1

    return roll_ret, S_t, C_t_new

def carry_forward():
    if last_known_C is None:
        return None
    return max(0, last_known_C + last_known_delta * (S_today - last_known_S))  # <- max(0, ...) added

In [37]:
# ── Preload full-year SPX close ── 12
spx_year = yf.download("^GSPC", start="2022-01-01", end="2023-01-05")["Close"].squeeze()
spx_year.index = spx_year.index.date
print(type(spx_year), spx_year.head())
print(f"Total trading days loaded: {len(spx_year)}")

[*********************100%***********************]  1 of 1 completed

<class 'pandas.Series'> 2022-01-03    4796.560059
2022-01-04    4793.540039
2022-01-05    4700.580078
2022-01-06    4696.049805
2022-01-07    4677.029785
Name: ^GSPC, dtype: float64
Total trading days loaded: 253


In [38]:
# ── compute_roll_leg: S_VWAV now trade-size-weighted, matching the SAME new-contract trades used for C_VWAP ──
def compute_roll_leg(roll_date, K_old, K_new, K_new_expiration, SOQ, C_settle, C_VWAP_new, S_prev, C_prev):
    fname = f"UnderlyingOptionsTradesCalcs_{roll_date.strftime('%Y-%m-%d')}.csv"
    fpath = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(fpath)
    df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

    new_mask = (
        (df["root"] == "SPX") & (df["option_type"] == "C")
        & (df["strike"] == K_new)
        & (df["expiration"] == K_new_expiration.strftime("%Y-%m-%d"))
    )
    new_day = df[new_mask].sort_values("quote_datetime")

    window_mask = (
        (new_day["quote_datetime"].dt.time >= pd.Timestamp("11:30:00").time())
        & (new_day["quote_datetime"].dt.time <= pd.Timestamp("13:30:00").time())
    )
    window_trades = new_day[window_mask]

    # trade-size-weighted underlying VWAP, same trades as the option VWAP
    S_VWAV = (window_trades["underlying_bid"] * window_trades["trade_size"]).sum() / window_trades["trade_size"].sum()

    before_4pm = new_day[new_day["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
    C_t_new = (before_4pm.iloc[-1]["best_bid"] + before_4pm.iloc[-1]["best_ask"]) / 2

    S_t = spx_year.loc[roll_date]

    Ra = (SOQ + 0 - C_settle) / (S_prev - C_prev) - 1
    Rb = S_VWAV / SOQ - 1
    Rc = (S_t - C_t_new) / (S_VWAV - C_VWAP_new) - 1
    roll_ret = (1 + Ra) * (1 + Rb) * (1 + Rc) - 1

    return roll_ret, S_t, C_t_new

In [39]:
# ── Cell 14: RESET baseline (always run this before Cell 15!) ──
jan21_file = os.path.join(DATA_DIR, "UnderlyingOptionsTradesCalcs_2022-01-21.csv")
jan21_df = pd.read_csv(jan21_file)
jan21_df["quote_datetime"] = pd.to_datetime(jan21_df["quote_datetime"])

mask = (
    (jan21_df["root"] == "SPX") & (jan21_df["option_type"] == "C")
    & (jan21_df["strike"] == 4475)
    & (jan21_df["expiration"] == "2022-02-18")
    & (~jan21_df["trade_condition_id"].isin(RARE_CODES))
)
jan21_opt = jan21_df[mask].sort_values("quote_datetime")
before_4pm = jan21_opt[jan21_opt["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
C_prev = (before_4pm.iloc[-1]["best_bid"] + before_4pm.iloc[-1]["best_ask"]) / 2
S_prev = spx_year.loc[pd.Timestamp("2022-01-21").date()]

print(f"RESET: S_prev={S_prev}, C_prev={C_prev}")
assert abs(S_prev - 4397.94) < 1, "S_prev not reset correctly!"
assert abs(C_prev - 80.45) < 1, "C_prev not reset correctly!"
print("Baseline reset confirmed OK.")

RESET: S_prev=4397.93994140625, C_prev=80.44999999999999
Baseline reset confirmed OK.


In [40]:
# ── Main loop with outlier flagging ──
all_returns = {}
diagnostics = []

for i in range(len(roll_dates) - 1):
    period_start = roll_dates.iloc[i]["roll_date"]
    period_end = roll_dates.iloc[i + 1]["roll_date"]
    held_strike = roll_dates.iloc[i]["K_new"]
    held_exp = roll_dates.iloc[i]["expiration"]

    print(f"Period {i}: {period_start} -> {period_end}, strike={held_strike}")

    non_roll_days = pd.bdate_range(period_start, period_end)[1:-1]
    opt_series = get_option_daily_series(held_strike, held_exp, non_roll_days.min(), non_roll_days.max())

    for _, row in opt_series.sort_values("date").iterrows():
        d = row["date"]
        S_t = spx_year.loc[d]
        C_t = row["C_t"]
        ret = (S_t - C_t) / (S_prev - C_prev) - 1
        all_returns[d] = ret
        diagnostics.append({"date": d, "S_t": S_t, "C_t": C_t, "S_prev": S_prev,
                             "C_prev": C_prev, "ret": ret, "n_quotes": row["n_quotes"],
                             "approx": row["approx"]})
        S_prev, C_prev = S_t, C_t

    row_next = roll_dates.iloc[i + 1]
    roll_ret, S_t, C_t = compute_roll_leg(
        roll_date=period_end,
        K_old=row_next["K_old"], K_new=row_next["K_new"],
        K_new_expiration=row_next["expiration"],
        SOQ=row_next["SOQ"], C_settle=row_next["C_settle"],
        C_VWAP_new=row_next["C_VWAP_new"],
        S_prev=S_prev, C_prev=C_prev,
    )
    all_returns[period_end] = roll_ret
    diagnostics.append({"date": period_end, "S_t": S_t, "C_t": C_t, "S_prev": S_prev,
                         "C_prev": C_prev, "ret": roll_ret, "n_quotes": None,
                         "approx": False, "is_roll": True})
    S_prev, C_prev = S_t, C_t

our_returns = pd.Series(all_returns).sort_index()
diag_df = pd.DataFrame(diagnostics)

print(f"\nTotal days reconstructed: {len(our_returns)}")
print(f"Approximated (intrinsic value) days: {diag_df['approx'].sum()}")

outliers = diag_df[diag_df["ret"].abs() > 0.03]
print(f"\n{len(outliers)} outlier day(s) with |ret| > 3%:")
print(outliers[["date", "S_t", "C_t", "S_prev", "C_prev", "ret", "n_quotes", "approx"]])

Period 0: 2022-01-21 -> 2022-02-18, strike=4475
Period 1: 2022-02-18 -> 2022-03-18, strike=4365
Period 2: 2022-03-18 -> 2022-04-14, strike=4415
Period 3: 2022-04-14 -> 2022-05-20, strike=4430
Period 4: 2022-05-20 -> 2022-06-17, strike=3885
Period 5: 2022-06-17 -> 2022-07-15, strike=3655
Period 6: 2022-07-15 -> 2022-08-19, strike=3850
Period 7: 2022-08-19 -> 2022-09-16, strike=4235
Period 8: 2022-09-16 -> 2022-10-21, strike=3855
Period 9: 2022-10-21 -> 2022-11-18, strike=3680
Period 10: 2022-11-18 -> 2022-12-16, strike=3950

Total days reconstructed: 228
Approximated (intrinsic value) days: 11

6 outlier day(s) with |ret| > 3%:
           date          S_t    C_t       S_prev    C_prev       ret  \
67   2022-04-29  4131.930176  9.350  4287.500000  30.70000 -0.031531   
71   2022-05-05  4146.870117  6.500  4300.169922  23.40000 -0.031893   
73   2022-05-09  3991.239990  1.075  4123.339844   3.05000 -0.031581   
80   2022-05-18  3923.679932  0.000  4088.850098   0.21168 -0.040346   
100  

In [41]:
# ── Compare full-year reconstruction against official BXM ──
our_df = our_returns.rename("our_ret").to_frame()
our_df.index.name = "date"

final_check = our_df.merge(
    bxm_full.set_index("date")[["bxm_ret"]], left_index=True, right_index=True, how="inner"
)
final_check["te"] = final_check["our_ret"] - final_check["bxm_ret"]

print(f"N days matched: {len(final_check)}")
print(f"Our ann. vol:         {final_check['our_ret'].std() * (252**0.5):.4%}")
print(f"BXM ann. vol:         {final_check['bxm_ret'].std() * (252**0.5):.4%}")
print(f"Full-year daily TE:   {final_check['te'].std() * (252**0.5):.4%}")
print(f"Mean daily TE:        {final_check['te'].mean():.4%}")

N days matched: 228
Our ann. vol:         17.2717%
BXM ann. vol:         16.6190%
Full-year daily TE:   3.4619%
Mean daily TE:        -0.0048%


In [42]:
# ── Find days where OUR reconstruction deviates most from actual BXM (final version) ──
final_check["abs_te"] = final_check["te"].abs()
worst_days = final_check.sort_values("abs_te", ascending=False).head(15)

print(worst_days[["our_ret", "bxm_ret", "te"]])
print(f"\nSum of squared te from top 15 worst days: {(worst_days['te']**2).sum():.6f}")
print(f"Sum of squared te from all {len(final_check)} days:       {(final_check['te']**2).sum():.6f}")
print(f"=> top 15 days account for {(worst_days['te']**2).sum() / (final_check['te']**2).sum():.1%} of total TE variance")

             our_ret   bxm_ret        te
date                                    
2022-11-11 -0.014942 -0.000565 -0.014377
2022-11-10  0.021664  0.010320  0.011344
2022-07-14  0.008130  0.000149  0.007981
2022-08-05  0.008007  0.000175  0.007832
2022-08-08 -0.007500  0.000066 -0.007566
2022-07-15 -0.002942  0.004454 -0.007396
2022-06-24  0.013142  0.006023  0.007119
2022-06-27 -0.004413  0.001979 -0.006392
2022-07-05 -0.004835  0.001081 -0.005917
2022-10-28  0.013094  0.007459  0.005635
2022-07-01  0.009325  0.004300  0.005024
2022-10-26 -0.005041 -0.000592 -0.004449
2022-03-23 -0.007612 -0.003374 -0.004238
2022-02-07  0.004250  0.000167  0.004082
2022-03-18  0.005759  0.009556 -0.003797

Sum of squared te from top 15 worst days: 0.000825
Sum of squared te from all 228 days:       0.001080
=> top 15 days account for 76.4% of total TE variance


In [42]:
# ── Quick isolated test: May 20 roll only ──
may20_row = roll_dates[roll_dates["roll_date"] == pd.Timestamp("2022-05-20").date()].iloc[0]

# get_option_daily_series with forward-fill to get the correct May 19 carried-forward value
prev_series = get_option_daily_series(
    strike=4430, expiration=pd.Timestamp("2022-05-20"),
    start_date=pd.Timestamp("2022-04-15"), end_date=pd.Timestamp("2022-05-19"),
)
print(prev_series.tail())

C_prev_may19 = prev_series.iloc[-1]["C_t"]
S_prev_may19 = spx_year.loc[pd.Timestamp("2022-05-19").date()]

print(f"\nMay 19 (forward-filled if needed): S={S_prev_may19}, C={C_prev_may19}")

roll_ret_fixed, S_t, C_t = compute_roll_leg(
    roll_date=may20_row["roll_date"],
    K_old=may20_row["K_old"], K_new=may20_row["K_new"],
    K_new_expiration=may20_row["expiration"],
    SOQ=may20_row["SOQ"], C_settle=may20_row["C_settle"],
    C_VWAP_new=may20_row["C_VWAP_new"],
    S_prev=S_prev_may19, C_prev=C_prev_may19,
)

bxm_may20 = bxm_full[bxm_full["date"] == pd.Timestamp("2022-05-20").date()]["bxm_ret"].values[0]

print(f"\nFixed roll return:   {roll_ret_fixed:.4%}")
print(f"Old (broken):        -3.2917%")
print(f"Official BXM:        {bxm_may20:.4%}")

          date   C_t  n_quotes  forward_filled
19  2022-05-13  0.15         1           False
20  2022-05-16  0.05         5           False
21  2022-05-17  0.05         0            True
22  2022-05-18  0.05         0            True
23  2022-05-19  0.05         0            True

May 19 (forward-filled if needed): S=3900.7900390625, C=0.05

Fixed roll return:   -0.6147%
Old (broken):        -3.2917%
Official BXM:        -0.6187%


In [35]:
# ── Debug: check what's actually in diag_df around May 2022 ──
print(diag_df["date"].dtype)
may_dates = diag_df[(diag_df["date"] >= pd.Timestamp("2022-05-10").date())
                     & (diag_df["date"] <= pd.Timestamp("2022-05-20").date())]
print(may_dates[["date", "ret"]])

object
          date       ret
73  2022-05-10  0.002596
74  2022-05-11 -0.016390
75  2022-05-12 -0.001296
76  2022-05-13  0.023890
77  2022-05-16 -0.003922
78  2022-05-20 -0.032917


In [36]:
# ── Debug: why are May 17-19 missing for strike=4430? ──
for d in ["2022-05-17", "2022-05-18", "2022-05-19"]:
    fname = f"UnderlyingOptionsTradesCalcs_{d}.csv"
    fpath = os.path.join(DATA_DIR, fname)
    print(f"\n{d}: file exists = {os.path.exists(fpath)}")
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        mask = (
            (df["root"] == "SPX") & (df["option_type"] == "C")
            & (df["strike"] == 4430)
            & (df["expiration"] == "2022-05-20")
        )
        print(f"  rows matching strike=4430, exp=2022-05-20: {mask.sum()}")
        print(f"  unique strikes present for SPX calls exp=2022-05-20: "
              f"{sorted(df[(df['root']=='SPX') & (df['option_type']=='C') & (df['expiration']=='2022-05-20')]['strike'].unique())[:20]}")


2022-05-17: file exists = True
  rows matching strike=4430, exp=2022-05-20: 0
  unique strikes present for SPX calls exp=2022-05-20: [np.float64(3600.0), np.float64(3650.0), np.float64(3740.0), np.float64(3850.0), np.float64(3865.0), np.float64(3870.0), np.float64(3880.0), np.float64(3885.0), np.float64(3895.0), np.float64(3900.0), np.float64(3905.0), np.float64(3910.0), np.float64(3915.0), np.float64(3920.0), np.float64(3925.0), np.float64(3935.0), np.float64(3945.0), np.float64(3950.0), np.float64(3960.0), np.float64(3965.0)]

2022-05-18: file exists = True
  rows matching strike=4430, exp=2022-05-20: 0
  unique strikes present for SPX calls exp=2022-05-20: [np.float64(200.0), np.float64(3100.0), np.float64(3620.0), np.float64(3625.0), np.float64(3700.0), np.float64(3740.0), np.float64(3765.0), np.float64(3800.0), np.float64(3870.0), np.float64(3875.0), np.float64(3880.0), np.float64(3890.0), np.float64(3895.0), np.float64(3900.0), np.float64(3905.0), np.float64(3910.0), np.float64(

In [ ]:
# ── Check: is the last-quote timestamp lagging 4pm on the worst days, and are they deep ITM? ── 19
worst_dates = worst_days.index.tolist()

check_rows = diag_df[diag_df["date"].isin(worst_dates)][
    ["date", "S_t", "C_t", "n_quotes"]
].copy()
print(check_rows)

# quick moneyness proxy: how deep ITM was the held contract on these days
# (strike varies by period, so pull from roll_dates per period for reference)
print("\nHeld strikes by period for context:")
print(roll_dates[["roll_date", "K_new"]])

           date          S_t     C_t  n_quotes
91   2022-06-03  4108.540039  300.40       0.0
92   2022-06-06  4121.430176  246.85      20.0
105  2022-06-24  3911.739990  256.90     146.0
106  2022-06-27  3900.110107  261.40       7.0
111  2022-07-05  3831.389893  190.40      27.0
118  2022-07-14  3790.379883  107.45       2.0
119  2022-07-15  3863.159912  112.60       NaN
134  2022-08-05  4145.189941  267.80       2.0
135  2022-08-08  4140.060059  291.75      23.0
193  2022-10-28  3901.060059  245.75     212.0
201  2022-11-09  3748.570068  136.00       0.0
202  2022-11-10  3956.370117  222.55       1.0
203  2022-11-11  3992.929932  314.90      21.0
204  2022-11-14  3957.250000  314.90       0.0
205  2022-11-15  3991.729980  314.90       0.0

Held strikes by period for context:
     roll_date  K_new
0   2022-01-21   4475
1   2022-02-18   4365
2   2022-03-18   4415
3   2022-04-14   4430
4   2022-05-20   3885
5   2022-06-17   3655
6   2022-07-15   3850
7   2022-08-19   4235
8   2022-09-1

In [21]:
# ── Inspect raw quotes around 4pm for a suspicious day pair (Jun 22-23) ──
for d in ["2022-06-22", "2022-06-23"]:
    fname = f"UnderlyingOptionsTradesCalcs_{d}.csv"
    fpath = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(fpath)
    df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

    mask = (
        (df["root"] == "SPX") & (df["option_type"] == "C")
        & (df["strike"] == 3655)
        & (df["expiration"] == "2022-07-15")
    )
    day_df = df[mask].sort_values("quote_datetime")

    print(f"\n=== {d} ===")
    print(f"Columns available: {df.columns.tolist()}")
    print(day_df.tail(10))  # last 10 quotes of the day, whatever time


=== 2022-06-22 ===
Columns available: ['underlying_symbol', 'market_date', 'quote_datetime', 'sequence_number', 'root', 'expiration', 'strike', 'option_type', 'exchange_id', 'trade_size', 'trade_price', 'trade_condition_id', 'canceled_trade_condition_id', 'best_bid', 'best_ask', 'trade_iv', 'trade_delta', 'trade_gamma', 'trade_vega', 'trade_theta', 'trade_rho', 'underlying_bid', 'underlying_ask']
Empty DataFrame
Columns: [underlying_symbol, market_date, quote_datetime, sequence_number, root, expiration, strike, option_type, exchange_id, trade_size, trade_price, trade_condition_id, canceled_trade_condition_id, best_bid, best_ask, trade_iv, trade_delta, trade_gamma, trade_vega, trade_theta, trade_rho, underlying_bid, underlying_ask]
Index: []

[0 rows x 23 columns]

=== 2022-06-23 ===
Columns available: ['underlying_symbol', 'market_date', 'quote_datetime', 'sequence_number', 'root', 'expiration', 'strike', 'option_type', 'exchange_id', 'trade_size', 'trade_price', 'trade_condition_id',

In [22]:
# ── Check timezone assumption: are quote_datetime values actually in ET? ──
fname = "UnderlyingOptionsTradesCalcs_2022-06-23.csv"
fpath = os.path.join(DATA_DIR, fname)
df = pd.read_csv(fpath)
df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

mask = (df["root"] == "SPX") & (df["option_type"] == "C") & (df["strike"] == 3655) & (df["expiration"] == "2022-07-15")
day_df = df[mask].sort_values("quote_datetime")

print("First 5 quotes of the day:")
print(day_df[["quote_datetime", "best_bid", "best_ask"]].head(5))
print("\nLast 5 quotes of the day:")
print(day_df[["quote_datetime", "best_bid", "best_ask"]].tail(5))
print(f"\nEarliest quote time: {day_df['quote_datetime'].min()}")
print(f"Latest quote time:   {day_df['quote_datetime'].max()}")

First 5 quotes of the day:
              quote_datetime  best_bid  best_ask
2718 2022-06-23 14:02:52.122     165.1     166.2
2719 2022-06-23 14:03:21.165     162.7     163.7
2720 2022-06-23 15:55:08.112     180.7     188.1
2721 2022-06-23 15:55:38.469     185.0     185.9
2722 2022-06-23 15:56:09.112     184.9     185.4

Last 5 quotes of the day:
              quote_datetime  best_bid  best_ask
2736 2022-06-23 16:03:27.061     181.8     182.9
2737 2022-06-23 16:03:51.517     181.2     182.3
2738 2022-06-23 16:03:51.517     181.2     182.3
2739 2022-06-23 17:22:37.712     170.3     182.3
2740 2022-06-23 17:22:37.716     170.3     182.3

Earliest quote time: 2022-06-23 14:02:52.122000
Latest quote time:   2022-06-23 17:22:37.716000


In [24]:
# ── Confirm timezone: check trade volume concentration by hour (raw quote_datetime, no conversion) ──
fname = "UnderlyingOptionsTradesCalcs_2022-06-17.csv"  # a roll date, should have a volume spike in the VWAP window
fpath = os.path.join(DATA_DIR, fname)
df = pd.read_csv(fpath)
df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

mask = (df["root"] == "SPX") & (df["option_type"] == "C") & (df["strike"] == 3655) & (df["expiration"] == "2022-07-15")
day_df = df[mask]

hourly_volume = day_df.groupby(day_df["quote_datetime"].dt.hour)["trade_size"].sum()
print("Trade volume by hour (raw quote_datetime hour, likely UTC):")
print(hourly_volume)
print("\nIf UTC: 11:30am-1:30pm ET = 15:30-17:30 UTC (summer/EDT)")
print("If already ET: 11:30am-1:30pm ET = hour 11-13")

Trade volume by hour (raw quote_datetime hour, likely UTC):
quote_datetime
10      20
11    1179
12    3507
13    1877
14     670
15     143
Name: trade_size, dtype: int64

If UTC: 11:30am-1:30pm ET = 15:30-17:30 UTC (summer/EDT)
If already ET: 11:30am-1:30pm ET = hour 11-13


In [25]:
# ── Check exactly what our pipeline computed for Jun 22-23 ──
check = diag_df[diag_df["date"].isin([pd.Timestamp("2022-06-22").date(), pd.Timestamp("2022-06-23").date()])]
print(check[["date", "S_t", "C_t", "S_prev", "C_prev", "ret", "n_quotes", "approx"]])

           date          S_t         C_t       S_prev      C_prev       ret  \
103  2022-06-22  3759.889893  104.889893  3764.790039  171.250000  0.017103   
104  2022-06-23  3795.729980  188.300000  3759.889893  104.889893 -0.013015   

     n_quotes  approx  
103       0.0    True  
104      12.0   False  


In [26]:
# ── Check what other files exist in the raw data folder — did we buy more than just Trades? ──
import os

all_files = os.listdir(DATA_DIR)
print(f"Total files in {DATA_DIR}: {len(all_files)}")
print("\nUnique filename patterns (prefix before date):")
prefixes = set(f.rsplit("_2022", 1)[0] for f in all_files if "2022" in f)
print(prefixes)

# also check the parent raw/ folder for other datasets entirely
parent_dir = os.path.dirname(DATA_DIR.rstrip("/"))
print(f"\nContents of {parent_dir}:")
print(os.listdir(parent_dir))

Total files in ../data/raw/cboe_spx_2022/: 251

Unique filename patterns (prefix before date):
{'UnderlyingOptionsTradesCalcs'}

Contents of ../data/raw:
['BXMD_Bloomberg_2022.csv', 'BXM_Bloomberg_2022.csv', 'BXM_History.csv', 'cboe_spx_2022', 'Equity EOD Summary', 'HistoricalData_PBP.csv', 'PBP_Bloomberg_2022.csv', 'SPX_Bloomberg_2022.csv', 'VIX_Bloomberg_2022.csv']


In [27]:
# ── Quick peek at BXM_Bloomberg_2022.csv and SPX_Bloomberg_2022.csv ──
bxm_bbg = pd.read_csv("../data/raw/BXM_Bloomberg_2022.csv")
print("BXM_Bloomberg_2022.csv:")
print(bxm_bbg.head())
print(bxm_bbg.columns.tolist())

print("\n" + "="*50 + "\n")

spx_bbg = pd.read_csv("../data/raw/SPX_Bloomberg_2022.csv", skiprows=6)
print("SPX_Bloomberg_2022.csv:")
print(spx_bbg.head())
print(spx_bbg.columns.tolist())

BXM_Bloomberg_2022.csv:
  Start Date    1/1/2022
0   End Date  12/31/2022
1        NaN         NaN
2        NaN   BXM Index
3        NaN  Last Price
4      Dates     PX_LAST
['Start Date', '1/1/2022']


SPX_Bloomberg_2022.csv:
    1/3/2022  4796.56  4778.14  4796.64  4758.17
0   1/4/2022  4793.54  4804.51  4818.62  4774.27
1   1/5/2022  4700.58  4787.99  4797.70  4699.44
2   1/6/2022  4696.05  4693.39  4725.01  4671.26
3   1/7/2022  4677.03  4697.66  4707.95  4662.74
4  1/10/2022  4670.29  4655.34  4673.02  4582.24
['1/3/2022', '4796.56', '4778.14', '4796.64', '4758.17']
